In [0]:
!pip install openpyxl

In [0]:
import pandas as pd


# Path to the Excel file in a Databricks volume (Unity Catalog or DBFS)
file_path = "/Volumes/insights_dev_003/dwh/rawdataforboth/FinActVsInsights-Tables_Columns.xlsx"

# Read the Excel file
df = pd.read_excel(file_path, sheet_name='FinAct', engine='openpyxl')  # Use engine='openpyxl' for .xlsx
df.columns = [col.strip().replace(' ', '_').replace('\n', '_') for col in df.columns]
display(df)


In [0]:
spark_df = spark.createDataFrame(df)

# Step 3: Write the Spark DataFrame to a Delta table in Unity Catalog
spark_df.write.format('delta').mode('overwrite').saveAsTable('insights_dev_003.finactdm.tbl_dim_policy_section')


In [0]:
%sql
DESCRIBE TABLE insights_dev_003.finactdm.tbl_dim_policy_section;


In [0]:
%sql
DESCRIBE TABLE insights_dev_003.dwh.dim_policy_section;


real work

In [0]:
%sql
-- schema of Finact ( source of truth )
WITH finact_schema AS (
    SELECT 
        table_name,
        column_name, 
        data_type
    FROM insights_dev_003.information_schema.columns
    WHERE table_schema = 'finactdm' 
      AND table_name = 'tbl_dim_policy_section'
),

-- schema of DWH ( insights )
dwh_schema AS (
    SELECT 
        table_name,
        column_name, 
        data_type
    FROM insights_dev_003.information_schema.columns
    WHERE table_schema = 'dwh' 
      AND table_name = 'dim_policy_section'
),

-- Comparing missing or mismatched in ( insights )
finact_vs_dwh AS (
    SELECT 
        f.column_name,
        f.data_type AS finact_type,
        d.data_type AS dwh_type,
        CASE 
            WHEN d.column_name IS NULL THEN 'Missing in DWH'
            WHEN f.data_type <> d.data_type THEN 'Data type mismatch'
            ELSE 'Unknown'
        END AS mismatch_reason
    FROM finact_schema f
    LEFT JOIN dwh_schema d
        ON f.column_name = d.column_name
       AND d.table_name = 'dim_policy_section'
    WHERE d.column_name IS NULL OR f.data_type <> d.data_type
),

-- Compare extra in insights
dwh_vs_finact AS (
    SELECT 
        d.column_name,
        NULL AS finact_type,
        d.data_type AS dwh_type,
        'Extra in DWH' AS mismatch_reason
    FROM dwh_schema d
    LEFT JOIN finact_schema f
        ON d.column_name = f.column_name
       AND f.table_name = 'tbl_dim_policy_section'
    WHERE f.column_name IS NULL
)

-- Final
SELECT * FROM finact_vs_dwh
UNION ALL
SELECT * FROM dwh_vs_finact;


column_name,finact_type,dwh_type,mismatch_reason
Cancellation_Date,STRING,null,Missing in DWH
Credit_Period,DOUBLE,LONG,Data type mismatch
Basis_of_Cover,STRING,null,Missing in DWH
Inward_Outward,STRING,null,Missing in DWH
Dummy_Policy_Flag,STRING,null,Missing in DWH
Enter_Status,STRING,null,Missing in DWH
RI_Type_Id,STRING,null,Missing in DWH
Estimated_Acquisition_Ratio,DOUBLE,null,Missing in DWH
Office_Code,STRING,null,Missing in DWH
Office_Name,STRING,null,Missing in DWH


Row wise comparison

explicit columns

In [0]:
%sql
SELECT 
    COALESCE(i.Policy_Section_Reference, f.Policy_Section_Reference) AS SectionRef,
    i.Policy_Header_Reference AS iHead,
    f.Policy_Header_Reference AS fHead,
    i.Inception_Date AS iDate,
    f.Inception_Date AS fDate,
    CASE 
        WHEN i.Policy_Section_Reference IS NULL THEN 'Finact only'
        WHEN f.Policy_Section_Reference IS NULL THEN 'Insights only'
        WHEN i.Policy_Header_Reference <> f.Policy_Header_Reference 
             OR i.Inception_Date <> f.Inception_Date THEN 'Mismatch'
        ELSE 'Same'
    END AS Status
FROM insights_dev_003.finactdm.tbl_dim_policy_section i
FULL JOIN insights_dev_003.dwh.dim_policy_section f
    ON i.Policy_Section_Reference = f.Policy_Section_Reference;


SectionRef,iHead,fHead,iDate,fDate,Status
-2326_CAN,-2326_CAN,null,Jan 1 2019 12:00AM,null,Insights only
000000XLBU_CAN,000000XLBU_CAN,null,May 1 2000 12:00AM,null,Insights only
000000XLCQ_CAN,000000XLCQ_CAN,null,Jan 1 2000 12:00AM,null,Insights only
000000XLDO_CAN,000000XLDO_CAN,null,Apr 1 2000 12:00AM,null,Insights only
000001QSAB_CAN,000001QSAB_CAN,null,Jan 1 2001 12:00AM,null,Insights only
000001XLBD_CAN,000001XLBD_CAN,null,Jan 1 2001 12:00AM,null,Insights only
000001XLCI_CAN,000001XLCI_CAN,null,Apr 1 2001 12:00AM,null,Insights only
000002XLAY_CAN,000002XLAY_CAN,null,Jan 1 2002 12:00AM,null,Insights only
000003QSAB_CAN,000003QSAB_CAN,null,Jan 1 2003 12:00AM,null,Insights only
000004XLCD_CAN,000004XLCD_CAN,null,Jan 1 2004 12:00AM,null,Insights only


sql plus python implementation

In [0]:
# Define table names
table1 = "insights_dev_003.finactdm.tbl_dim_policy_section"
table2 = "insights_dev_003.dwh.dim_policy_section"
join_key = "Policy_Section_Reference"

# Step 1: Get common columns from both tables (excluding join key)
columns1 = [field.name for field in spark.table(table1).schema]
columns2 = [field.name for field in spark.table(table2).schema]
common_columns = list(set(columns1) & set(columns2))
if join_key in common_columns:
    common_columns.remove(join_key)

# Step 2: Create column select and comparison strings
column_selects = ",\n        ".join([
    f"i.{col} AS i_{col}, f.{col} AS f_{col}" for col in common_columns
])
comparison_conditions = " OR\n            ".join([
    f"i_{col} IS DISTINCT FROM f_{col}" for col in common_columns
])

# Step 3: Construct SQL query
final_sql = f"""
WITH CombinedData AS (
    SELECT 
        COALESCE(i.{join_key}, f.{join_key}) AS {join_key},
        i.{join_key} AS i_{join_key},
        f.{join_key} AS f_{join_key},
        {column_selects}
    FROM {table1} i
    FULL OUTER JOIN {table2} f
        ON i.{join_key} = f.{join_key}
),
Comparison AS (
    SELECT 
        {join_key},
        CASE 
            WHEN i_{join_key} IS NULL THEN 'Missing in Finact'
            WHEN f_{join_key} IS NULL THEN 'Missing in Insights'
            WHEN {comparison_conditions} THEN 'Mismatch in Column Values'
            ELSE 'Exact Match'
        END AS Status
    FROM CombinedData
)
SELECT * FROM Comparison
"""

# Step 4: Run and display result
display(spark.sql(final_sql))


Policy_Section_Reference,Status
-2326_CAN,Missing in Insights
000000XLBU_CAN,Missing in Insights
000000XLCQ_CAN,Missing in Insights
000000XLDO_CAN,Missing in Insights
000001QSAB_CAN,Missing in Insights
000001XLBD_CAN,Missing in Insights
000001XLCI_CAN,Missing in Insights
000002XLAY_CAN,Missing in Insights
000003QSAB_CAN,Missing in Insights
000004XLCD_CAN,Missing in Insights


In [0]:
# %sql
# WITH table1_cols AS (
#     SELECT column_name
#     FROM insights_dev_003.information_schema.columns
#     WHERE table_schema = 'finactdm'
#       AND table_name = 'tbl_dim_policy_section'
# ),
# table2_cols AS (
#     SELECT column_name
#     FROM insights_dev_003.information_schema.columns
#     WHERE table_schema = 'dwh'
#       AND table_name = 'dim_policy_section'
# ),
# common_cols AS (
#     SELECT t1.column_name
#     FROM table1_cols t1
#     INNER JOIN table2_cols t2
#         ON t1.column_name = t2.column_name
#     WHERE t1.column_name != 'Policy_Section_Reference'
# ),
# expressions AS (
#     SELECT
#         concat_ws(',\n    ',
#             collect_list('i.' || column_name || ' AS i_' || column_name || ', f.' || column_name || ' AS f_' || column_name)
#         ) AS column_selects,
#         concat_ws(' OR\n    ',
#             collect_list('i_' || column_name || ' IS DISTINCT FROM f_' || column_name)
#         ) AS comparison_conditions
#     FROM common_cols
# )
# SELECT '
# WITH CombinedData AS (
#     SELECT 
#         COALESCE(i.Policy_Section_Reference, f.Policy_Section_Reference) AS Policy_Section_Reference,
#         i.Policy_Section_Reference AS i_Policy_Section_Reference,
#         f.Policy_Section_Reference AS f_Policy_Section_Reference,
#         ' || column_selects || '
#     FROM insights_dev_003.finactdm.tbl_dim_policy_section i
#     FULL OUTER JOIN insights_dev_003.dwh.dim_policy_section f
#         ON i.Policy_Section_Reference = f.Policy_Section_Reference
# ),
# Comparison AS (
#     SELECT 
#         Policy_Section_Reference,
#         CASE 
#             WHEN i_Policy_Section_Reference IS NULL THEN ''Missing in Finact''
#             WHEN f_Policy_Section_Reference IS NULL THEN ''Missing in Insights''
#             WHEN ' || comparison_conditions || ' THEN ''Mismatch in Column Values''
#             ELSE ''Exact Match''
#         END AS Status
#     FROM CombinedData
# )
# SELECT * FROM Comparison;' AS final_sql
# FROM expressions;


final_sql
"WITH CombinedData AS ( SELECT COALESCE(i.Policy_Section_Reference, f.Policy_Section_Reference) AS Policy_Section_Reference, i.Policy_Section_Reference AS i_Policy_Section_Reference, f.Policy_Section_Reference AS f_Policy_Section_Reference, i.Aggregated_Data_Input_Policy_Indicator AS i_Aggregated_Data_Input_Policy_Indicator, f.Aggregated_Data_Input_Policy_Indicator AS f_Aggregated_Data_Input_Policy_Indicator, i.Attachment_Priority AS i_Attachment_Priority, f.Attachment_Priority AS f_Attachment_Priority, i.Bulk_Policy_Indicator AS i_Bulk_Policy_Indicator, f.Bulk_Policy_Indicator AS f_Bulk_Policy_Indicator, i.Credit_Period AS i_Credit_Period, f.Credit_Period AS f_Credit_Period, i.Expiry_Date AS i_Expiry_Date, f.Expiry_Date AS f_Expiry_Date, i.Inception_Date AS i_Inception_Date, f.Inception_Date AS f_Inception_Date, i.Long_Term_Agreement_Expiry_Date AS i_Long_Term_Agreement_Expiry_Date, f.Long_Term_Agreement_Expiry_Date AS f_Long_Term_Agreement_Expiry_Date, i.Notice_Period AS i_Notice_Period, f.Notice_Period AS f_Notice_Period, i.Novated_Policy_Indicator AS i_Novated_Policy_Indicator, f.Novated_Policy_Indicator AS f_Novated_Policy_Indicator, i.Payment_Assignment_Policy_Indicator AS i_Payment_Assignment_Policy_Indicator, f.Payment_Assignment_Policy_Indicator AS f_Payment_Assignment_Policy_Indicator, i.Policy_Header_Reference AS i_Policy_Header_Reference, f.Policy_Header_Reference AS f_Policy_Header_Reference, i.Policy_Reference AS i_Policy_Reference, f.Policy_Reference AS f_Policy_Reference, i.Renewable_Indicator AS i_Renewable_Indicator, f.Renewable_Indicator AS f_Renewable_Indicator, i.Settlement_Frequency AS i_Settlement_Frequency, f.Settlement_Frequency AS f_Settlement_Frequency, i.Sub_Class_Code AS i_Sub_Class_Code, f.Sub_Class_Code AS f_Sub_Class_Code, i.Unrecognised_External_Policy_Indicator AS i_Unrecognised_External_Policy_Indicator, f.Unrecognised_External_Policy_Indicator AS f_Unrecognised_External_Policy_Indicator, i.Year_Of_Account AS i_Year_Of_Account, f.Year_Of_Account AS f_Year_Of_Account FROM insights_dev_003.finactdm.tbl_dim_policy_section i FULL OUTER JOIN insights_dev_003.dwh.dim_policy_section f ON i.Policy_Section_Reference = f.Policy_Section_Reference ), Comparison AS ( SELECT Policy_Section_Reference, CASE WHEN i_Policy_Section_Reference IS NULL THEN Missing in Finact WHEN f_Policy_Section_Reference IS NULL THEN Missing in Insights WHEN i_Aggregated_Data_Input_Policy_Indicator IS DISTINCT FROM f_Aggregated_Data_Input_Policy_Indicator OR i_Attachment_Priority IS DISTINCT FROM f_Attachment_Priority OR i_Bulk_Policy_Indicator IS DISTINCT FROM f_Bulk_Policy_Indicator OR i_Credit_Period IS DISTINCT FROM f_Credit_Period OR i_Expiry_Date IS DISTINCT FROM f_Expiry_Date OR i_Inception_Date IS DISTINCT FROM f_Inception_Date OR i_Long_Term_Agreement_Expiry_Date IS DISTINCT FROM f_Long_Term_Agreement_Expiry_Date OR i_Notice_Period IS DISTINCT FROM f_Notice_Period OR i_Novated_Policy_Indicator IS DISTINCT FROM f_Novated_Policy_Indicator OR i_Payment_Assignment_Policy_Indicator IS DISTINCT FROM f_Payment_Assignment_Policy_Indicator OR i_Policy_Header_Reference IS DISTINCT FROM f_Policy_Header_Reference OR i_Policy_Reference IS DISTINCT FROM f_Policy_Reference OR i_Renewable_Indicator IS DISTINCT FROM f_Renewable_Indicator OR i_Settlement_Frequency IS DISTINCT FROM f_Settlement_Frequency OR i_Sub_Class_Code IS DISTINCT FROM f_Sub_Class_Code OR i_Unrecognised_External_Policy_Indicator IS DISTINCT FROM f_Unrecognised_External_Policy_Indicator OR i_Year_Of_Account IS DISTINCT FROM f_Year_Of_Account THEN Mismatch in Column Values ELSE Exact Match END AS Status FROM CombinedData ) SELECT * FROM Comparison;"


In [0]:
# %sql

# WITH CombinedData AS (
#     SELECT 
#         COALESCE(i.Policy_Section_Reference, f.Policy_Section_Reference) AS Policy_Section_Reference,
#         i.Policy_Section_Reference AS i_Policy_Section_Reference,
#         f.Policy_Section_Reference AS f_Policy_Section_Reference,
#         i.Aggregated_Data_Input_Policy_Indicator AS i_Aggregated_Data_Input_Policy_Indicator, f.Aggregated_Data_Input_Policy_Indicator AS f_Aggregated_Data_Input_Policy_Indicator,
#     i.Attachment_Priority AS i_Attachment_Priority, f.Attachment_Priority AS f_Attachment_Priority,
#     i.Bulk_Policy_Indicator AS i_Bulk_Policy_Indicator, f.Bulk_Policy_Indicator AS f_Bulk_Policy_Indicator,
#     i.Credit_Period AS i_Credit_Period, f.Credit_Period AS f_Credit_Period,
#     i.Expiry_Date AS i_Expiry_Date, f.Expiry_Date AS f_Expiry_Date,
#     i.Inception_Date AS i_Inception_Date, f.Inception_Date AS f_Inception_Date,
#     i.Long_Term_Agreement_Expiry_Date AS i_Long_Term_Agreement_Expiry_Date, f.Long_Term_Agreement_Expiry_Date AS f_Long_Term_Agreement_Expiry_Date,
#     i.Notice_Period AS i_Notice_Period, f.Notice_Period AS f_Notice_Period,
#     i.Novated_Policy_Indicator AS i_Novated_Policy_Indicator, f.Novated_Policy_Indicator AS f_Novated_Policy_Indicator,
#     i.Payment_Assignment_Policy_Indicator AS i_Payment_Assignment_Policy_Indicator, f.Payment_Assignment_Policy_Indicator AS f_Payment_Assignment_Policy_Indicator,
#     i.Policy_Header_Reference AS i_Policy_Header_Reference, f.Policy_Header_Reference AS f_Policy_Header_Reference,
#     i.Policy_Reference AS i_Policy_Reference, f.Policy_Reference AS f_Policy_Reference,
#     i.Renewable_Indicator AS i_Renewable_Indicator, f.Renewable_Indicator AS f_Renewable_Indicator,
#     i.Settlement_Frequency AS i_Settlement_Frequency, f.Settlement_Frequency AS f_Settlement_Frequency,
#     i.Sub_Class_Code AS i_Sub_Class_Code, f.Sub_Class_Code AS f_Sub_Class_Code,
#     i.Unrecognised_External_Policy_Indicator AS i_Unrecognised_External_Policy_Indicator, f.Unrecognised_External_Policy_Indicator AS f_Unrecognised_External_Policy_Indicator,
#     i.Year_Of_Account AS i_Year_Of_Account, f.Year_Of_Account AS f_Year_Of_Account
#     FROM insights_dev_003.finactdm.tbl_dim_policy_section i
#     FULL OUTER JOIN insights_dev_003.dwh.dim_policy_section f
#         ON i.Policy_Section_Reference = f.Policy_Section_Reference
# ),
# Comparison AS (
#     SELECT 
#         Policy_Section_Reference,
#         CASE 
#             WHEN i_Policy_Section_Reference IS NULL THEN Missing in Finact
#             WHEN f_Policy_Section_Reference IS NULL THEN Missing in Insights
#             WHEN i_Aggregated_Data_Input_Policy_Indicator IS DISTINCT FROM f_Aggregated_Data_Input_Policy_Indicator OR
#     i_Attachment_Priority IS DISTINCT FROM f_Attachment_Priority OR
#     i_Bulk_Policy_Indicator IS DISTINCT FROM f_Bulk_Policy_Indicator OR
#     i_Credit_Period IS DISTINCT FROM f_Credit_Period OR
#     i_Expiry_Date IS DISTINCT FROM f_Expiry_Date OR
#     i_Inception_Date IS DISTINCT FROM f_Inception_Date OR
#     i_Long_Term_Agreement_Expiry_Date IS DISTINCT FROM f_Long_Term_Agreement_Expiry_Date OR
#     i_Notice_Period IS DISTINCT FROM f_Notice_Period OR
#     i_Novated_Policy_Indicator IS DISTINCT FROM f_Novated_Policy_Indicator OR
#     i_Payment_Assignment_Policy_Indicator IS DISTINCT FROM f_Payment_Assignment_Policy_Indicator OR
#     i_Policy_Header_Reference IS DISTINCT FROM f_Policy_Header_Reference OR
#     i_Policy_Reference IS DISTINCT FROM f_Policy_Reference OR
#     i_Renewable_Indicator IS DISTINCT FROM f_Renewable_Indicator OR
#     i_Settlement_Frequency IS DISTINCT FROM f_Settlement_Frequency OR
#     i_Sub_Class_Code IS DISTINCT FROM f_Sub_Class_Code OR
#     i_Unrecognised_External_Policy_Indicator IS DISTINCT FROM f_Unrecognised_External_Policy_Indicator OR
#     i_Year_Of_Account IS DISTINCT FROM f_Year_Of_Account THEN Mismatch in Column Values
#             ELSE Exact Match
#         END AS Status
#     FROM CombinedData
# )
# SELECT * FROM Comparison;

Sql implementation with dynamic columns

In [0]:
%sql
WITH CombinedData AS (
    SELECT 
        COALESCE(i.Policy_Section_Reference, f.Policy_Section_Reference) AS Policy_Section_Reference,
        CASE 
            WHEN i.Policy_Section_Reference IS NULL THEN 'Missing in Finact'
            WHEN f.Policy_Section_Reference IS NULL THEN 'Missing in Insights'
            WHEN HASH(i.*) <> HASH(f.*) THEN 'Mismatch in Column Values'
            ELSE 'Exact Match'
        END AS Status
    FROM insights_dev_003.finactdm.tbl_dim_policy_section i
    FULL OUTER JOIN insights_dev_003.dwh.dim_policy_section f
        ON i.Policy_Section_Reference = f.Policy_Section_Reference
)
SELECT * 
FROM CombinedData;


Policy_Section_Reference,Status
-2326_CAN,Missing in Insights
000000XLBU_CAN,Missing in Insights
000000XLCQ_CAN,Missing in Insights
000000XLDO_CAN,Missing in Insights
000001QSAB_CAN,Missing in Insights
000001XLBD_CAN,Missing in Insights
000001XLCI_CAN,Missing in Insights
000002XLAY_CAN,Missing in Insights
000003QSAB_CAN,Missing in Insights
000004XLCD_CAN,Missing in Insights
